# 🌧️ ICON-CH1 — Animated Hourly Rain Map (with real timestamps)

Renders an interactive Leaflet map with a time slider showing both `+Xh` and the actual forecast valid time.

## Cell 1 — Pre-render all 33 frames (run this first)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import base64, io, xarray as xr
import pandas as pd

ds = xr.open_dataset('icon_ch1_TOT_PREC_all_lead_times.nc')
hourly_rain = ds['hourly_rain']  # shape: (lead_time=33, y=295, x=429)

# Extract ref_time to compute real valid times
ref_time = pd.to_datetime(ds['ref_time'].values[0])
print(f'Model run: {ref_time.strftime("%Y-%m-%d %H:%M UTC")}')

# Distance mask
CH_LAT, CH_LON = 46.8, 8.2
lons = np.linspace(-0.817, 18.183, 429)
lats = np.linspace(41.183, 51.183, 295)
LON, LAT = np.meshgrid(lons, lats)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = np.radians(lat2-lat1), np.radians(lon2-lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

within_mask = haversine(LAT, LON, CH_LAT, CH_LON) <= 350

global_max = float(hourly_rain.max())
norm = mcolors.Normalize(vmin=0, vmax=max(global_max, 0.1))
cmap = plt.cm.YlOrRd

frames = []
n_hours = hourly_rain.shape[0]  # 33

for h in range(n_hours):
    rain_h = hourly_rain.isel(lead_time=h).values
    rain_masked = np.where(within_mask, rain_h, np.nan)
    rgba = cmap(norm(np.nan_to_num(rain_masked)))
    rgba[..., 3] = np.where(within_mask & (rain_masked >= 0.01), 1.0, 0)
    buf = io.BytesIO()
    plt.imsave(buf, np.flipud(rgba), format='png')
    buf.seek(0)
    frames.append('data:image/png;base64,' + base64.b64encode(buf.read()).decode('utf-8'))
    print(f'  rendered +{h+1:02d}h')

print(f'\nAll {n_hours} frames ready. Global max: {global_max:.2f} mm/m²')

Model run: 2026-04-05 06:00 UTC
  rendered +01h
  rendered +02h
  rendered +03h
  rendered +04h
  rendered +05h
  rendered +06h
  rendered +07h
  rendered +08h
  rendered +09h
  rendered +10h
  rendered +11h
  rendered +12h
  rendered +13h
  rendered +14h
  rendered +15h
  rendered +16h
  rendered +17h
  rendered +18h
  rendered +19h
  rendered +20h
  rendered +21h
  rendered +22h
  rendered +23h
  rendered +24h
  rendered +25h
  rendered +26h
  rendered +27h
  rendered +28h
  rendered +29h
  rendered +30h
  rendered +31h
  rendered +32h
  rendered +33h
  rendered +34h

All 34 frames ready. Global max: 4.96 mm/m²


## Cell 2 — Build HTML with time slider + real timestamps

In [2]:
# Build list of label strings: '+01h  Sun 06 Apr  07:00' etc.
time_labels = []
for h in range(1, n_hours + 1):
    valid = ref_time + pd.Timedelta(hours=h)
    label = f'+{h:02d}h  ·  {valid.strftime("%a %d %b  %H:%M")} UTC'
    time_labels.append(label)

frames_js = '[' + ','.join(f'"{f}"' for f in frames) + ']'
labels_js = '[' + ','.join(f'"{l}"' for l in time_labels) + ']'
global_max_js = round(global_max, 2)
run_label = ref_time.strftime('%Y-%m-%d %H:%M UTC')

html = f"""<!DOCTYPE html>
<html><head>
<meta charset="UTF-8">
<title>ICON-CH1 Hourly Rain</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ font-family:system-ui,sans-serif; background:#1a1a1a; color:#eee; }}
  #map {{ width:100vw; height:100vh; }}

  #controls-top {{
    position:absolute; top:12px; left:50%; transform:translateX(-50%);
    z-index:1000; background:rgba(20,20,20,0.85); backdrop-filter:blur(8px);
    border-radius:10px; padding:10px 18px; display:flex; align-items:center; gap:16px;
    border:1px solid rgba(255,255,255,0.1);
  }}

  #controls-bottom {{
    position:absolute; bottom:16px; left:50%; transform:translateX(-50%);
    z-index:1000; background:rgba(20,20,20,0.85); backdrop-filter:blur(8px);
    border-radius:10px; padding:12px 22px;
    border:1px solid rgba(255,255,255,0.1);
    display:flex; flex-direction:column; align-items:center; gap:8px; min-width:480px;
  }}
  #time-row {{ display:flex; align-items:center; gap:14px; width:100%; }}
  #time-slider {{ flex:1; accent-color:#4f98a3; }}
  #time-label {{
    font-size:13px; font-weight:600; color:#4f98a3;
    white-space:nowrap; min-width:48px;
  }}
  #time-display {{
    font-size:12px; color:#888; letter-spacing:0.02em;
  }}

  label {{ font-size:13px; color:#aaa; white-space:nowrap; }}
  #opacity-slider {{ width:100px; accent-color:#4f98a3; }}
  #opacity-val {{ font-size:13px; color:#4f98a3; min-width:30px; }}

  #legend {{
    position:absolute; bottom:110px; right:12px; z-index:1000;
    background:rgba(20,20,20,0.85); backdrop-filter:blur(8px);
    border-radius:10px; padding:12px 16px;
    border:1px solid rgba(255,255,255,0.1); min-width:160px;
  }}
  #legend h4 {{ font-size:11px; color:#aaa; margin-bottom:8px; text-transform:uppercase; letter-spacing:0.05em; }}
  .legend-bar {{
    height:12px; border-radius:4px; margin-bottom:4px;
    background:linear-gradient(to right,#ffffb2,#fed976,#feb24c,#fd8d3c,#f03b20,#bd0026);
  }}
  .legend-labels {{ display:flex; justify-content:space-between; font-size:11px; color:#888; }}
  #run-info {{ font-size:10px; color:#555; margin-top:6px; }}
</style>
</head><body>
<div id="map"></div>

<div id="controls-top">
  <label>Opacity</label>
  <input type="range" id="opacity-slider" min="0" max="100" value="65">
  <span id="opacity-val">65%</span>
</div>

<div id="controls-bottom">
  <div id="time-display">{time_labels[0]}</div>
  <div id="time-row">
    <label>+01h</label>
    <input type="range" id="time-slider" min="1" max="{n_hours}" value="1" step="1">
    <label>+{n_hours:02d}h</label>
  </div>
</div>

<div id="legend">
  <h4>Hourly rain</h4>
  <div class="legend-bar"></div>
  <div class="legend-labels"><span>0 mm</span><span>{global_max_js} mm</span></div>
  <div id="run-info">Run: {run_label}</div>
</div>

<script>
const FRAMES = {frames_js};
const LABELS = {labels_js};

const map = L.map('map').setView([46.5, 8.5], 7);

L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_nolabels/{{z}}/{{x}}/{{y}}{{r}}.png', {{
  attribution:'© OpenStreetMap contributors © CARTO',
  subdomains:'abcd', maxZoom:19
}}).addTo(map);

const bounds = [[41.183, -0.817], [51.183, 18.183]];
const rainOverlay = L.imageOverlay(FRAMES[0], bounds, {{opacity:0.65, interactive:false}}).addTo(map);

const labelsPane = map.createPane('labelsPane');
labelsPane.style.zIndex = 650;
labelsPane.style.pointerEvents = 'none';
L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_only_labels/{{z}}/{{x}}/{{y}}{{r}}.png', {{
  subdomains:'abcd', maxZoom:19, pane:'labelsPane'
}}).addTo(map);

document.getElementById('opacity-slider').addEventListener('input', function() {{
  rainOverlay.setOpacity(this.value / 100);
  document.getElementById('opacity-val').textContent = this.value + '%';
}});

document.getElementById('time-slider').addEventListener('input', function() {{
  const h = parseInt(this.value);
  rainOverlay.setUrl(FRAMES[h - 1]);
  document.getElementById('time-display').textContent = LABELS[h - 1];
}});
</script>
</body></html>"""

output_path = '/home/ignaz/meteodata-lab-env/rain_leaflet_animated_superslider.html'
with open(output_path, 'w') as f:
    f.write(html)
print(f'Saved! Open: file://{output_path}')

Saved! Open: file:///home/ignaz/meteodata-lab-env/rain_leaflet_animated_superslider.html
